In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from keras import initializers
import tensorflow as tf

PREDICTORS = ["PwmD", "PwmE"]   
TARGET_INT = ["Theta"]  
TARGET = ["DeltaTheta"]
     
INPUT_SIZE = len(PREDICTORS)  
OUTPUT_SIZE = len(TARGET)   
     
TIME_STEPS = 3
TS = 0.07

In [ ]:
Datasets = []
for i in range(4):
    Dataset = pd.read_excel(f"./../00-RotedData/Data.xlsx", f"D{i+1}")   
    Datasets.append(Dataset)

for i in range(4):   
    Dataset = pd.read_csv(f"./../00-Data/Data{i + 1}.csv")  
    Datasets.append(Dataset)
    
    
for i in range(len(Datasets)):
    Dataset = Datasets[i].copy()

    for var in TARGET_INT:
        Dataset[f"Delta{var}"] = (Dataset[var].shift(-1) - Dataset[var]) / TS
            
    Dataset = Dataset.dropna(subset=[f"Delta{var}" for var in TARGET_INT])

    Datasets[i] = Dataset

In [ ]:
import joblib
import os
os.makedirs("./scalers", exist_ok=True)

NormDatasets = []

SCALER = StandardScaler()
OUT_SCALER = StandardScaler()

Train1 = Datasets[0].copy()
Train1[PREDICTORS] = SCALER.fit_transform(Train1[PREDICTORS])
Train1[TARGET] = OUT_SCALER.fit_transform(Train1[TARGET])
NormDatasets.append(Train1)

Train2 = Datasets[1].copy()
Train2[PREDICTORS] = SCALER.transform(Train2[PREDICTORS])
Train2[TARGET] = OUT_SCALER.transform(Train2[TARGET])
NormDatasets.append(Train2)

# concatena
Train = pd.concat([Train1, Train2], ignore_index=True)

for i in range(4):
    CurrentTestDataset = Datasets[i + 2].copy()
    CurrentTestDataset[PREDICTORS] = SCALER.transform(CurrentTestDataset[PREDICTORS])
    CurrentTestDataset[TARGET] = OUT_SCALER.transform(CurrentTestDataset[TARGET])
    NormDatasets.append(CurrentTestDataset)
    
joblib.dump(SCALER, "./scalers/scaler.pkl")
joblib.dump(OUT_SCALER, "./scalers/out_scaler.pkl")

mean_tf = tf.constant(OUT_SCALER.mean_[0], dtype=tf.float32)
std_tf  = tf.constant(OUT_SCALER.scale_[0], dtype=tf.float32)

In [5]:
def CreateSequences(input_data, target_data, timesteps):
    X_seq, Y_seq = [], []
    
    for i in range(timesteps, len(input_data)):
        X_seq.append(input_data.iloc[i-timesteps:i].values)
        Y_seq.append(target_data.iloc[i])
    return np.array(X_seq), np.array(Y_seq)

x_train, y_train = CreateSequences(Train[PREDICTORS], Train[TARGET], TIME_STEPS)

x_val, y_val = CreateSequences((NormDatasets[5])[PREDICTORS], (NormDatasets[5])[TARGET], TIME_STEPS)
print(f"Dimensão da entrada: {np.shape(x_train)}")
print(f"Dimensão da saida: {np.shape(y_train)}")

print(f"Dimensão da entrada: {np.shape(x_val)}")
print(f"Dimensão da saida: {np.shape(y_val)}")

Dimensão da entrada: (1949, 3, 2)
Dimensão da saida: (1949, 1)
Dimensão da entrada: (1268, 3, 2)
Dimensão da saida: (1268, 1)


In [6]:
Wd_train =  Train["Wd"].values[:len(x_train)]
We_train =  Train["We"].values[:len(x_train)]
theta_train = Train["Theta"].values[:len(x_train)]

print(f"Dimensão da entrada: {np.shape(x_train)}")
print(f"Dimensão da saida: {np.shape(y_train)}")

print(f"Dimensão da entrada fisica : {np.shape(Wd_train)}")
print(f"Dimensão da entrada fisica: {np.shape(We_train)}")

Dimensão da entrada: (1949, 3, 2)
Dimensão da saida: (1949, 1)
Dimensão da entrada fisica : (1949,)
Dimensão da entrada fisica: (1949,)


In [ ]:
R = tf.constant(0.0328, dtype=tf.float32)
L = tf.constant(0.0615, dtype=tf.float32)
dt = tf.constant(TS, dtype=tf.float32)

def CinematicModel(Wd, We, theta):

    dtheta_cin = (R / (2 * L)) * (Wd - We)
    # dx_cin = (R / 2) * tf.cos(theta) * (Wd + We)
    # dy_cin = (R / 2) * tf.sin(theta) * (Wd + We)

    # return [dtheta_cin, dx_cin, dy_cin]
    return [dtheta_cin]
    

In [8]:

def NumericalIntegration(dataset, dq):

    q = [None] * OUTPUT_SIZE

    init_vals = np.array([
        dataset[name].iloc[0] for name in TARGET_INT
    ])

    for j in range(OUTPUT_SIZE):
        q[j] = init_vals[j] + np.cumsum(dq[j] * TS)

    return q

def GetCin(dataset): 
    dq = CinematicModel(tf.convert_to_tensor(dataset["Wd"].values, dtype=tf.float32),
                        tf.convert_to_tensor(dataset["We"].values, dtype=tf.float32), 
                        tf.convert_to_tensor(dataset["Theta"].values, dtype=tf.float32))
    q = NumericalIntegration(dataset, dq)
    return np.vstack(q).T

In [9]:
class PINN(tf.keras.Model):

    def __init__(self, architecture, initializer, regularizer):
        super(PINN, self).__init__()

        self.architecture = architecture  
        self.rnn_layers = []

        for i, units in enumerate(architecture):

            if i < len(architecture) - 1:
                self.rnn_layers.append(
                    tf.keras.layers.SimpleRNN(
                        units,
                        activation='tanh',
                        return_sequences=True,
                        kernel_initializer=initializer,
                        kernel_regularizer=regularizer,
                        recurrent_regularizer=regularizer,
                        bias_regularizer=regularizer
                    )
                )
            else:
                self.rnn_layers.append(
                    tf.keras.layers.SimpleRNN(
                        units,
                        activation='tanh',
                        kernel_initializer=initializer,
                        kernel_regularizer=regularizer,
                        recurrent_regularizer=regularizer,
                        bias_regularizer=regularizer
                    )
                )

        self.out_layer = tf.keras.layers.Dense(
            OUTPUT_SIZE,
            activation="linear",
            kernel_initializer=initializer,
            kernel_regularizer=regularizer,
            bias_regularizer=regularizer
        )

    def call(self, inputs):
        x = inputs
        for rnn in self.rnn_layers:
            x = rnn(x)
        return self.out_layer(x)

In [10]:
def PINN_to_RNN(pinn_model, time_steps, input_size):

    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Input(shape=(time_steps, input_size)))

    # recria arquitetura
    for layer in pinn_model.rnn_layers:
        model.add(
            tf.keras.layers.SimpleRNN(
                units=layer.units,
                activation='tanh',
                return_sequences=layer.return_sequences
            )
        )

    # camada final
    model.add(
        tf.keras.layers.Dense(
            pinn_model.out_layer.units,
            activation='linear'
        )
    )

    model.build((None, time_steps, input_size))

    model.set_weights(pinn_model.get_weights())

    return model

In [11]:
@tf.function
def train_step(model, optimizer, x, y, Wd, We, theta, Ld, Lp):

    with tf.GradientTape() as tape:

        pred = model(x, training=True)

        # loss dos dados
        data_loss = tf.reduce_mean(tf.square(pred - y))
        
        # termo físico
        physics = tf.stack(CinematicModel(Wd, We, theta), axis=1)   
             
        # normalização correta
        physics_denorm = (physics - mean_tf) / std_tf

        physics_loss = tf.reduce_mean(tf.square(pred - physics_denorm))

        loss = Ld * data_loss +  Lp * physics_loss


    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))

    return loss

In [12]:
Wd_train = tf.convert_to_tensor(Wd_train, dtype=tf.float32)
We_train = tf.convert_to_tensor(We_train, dtype=tf.float32)
theta_train = tf.convert_to_tensor(theta_train, dtype=tf.float32)

x_train_tf = tf.convert_to_tensor(x_train, dtype=tf.float32)
y_train_tf = tf.convert_to_tensor(y_train, dtype=tf.float32)

x_val_tf = tf.convert_to_tensor(x_val, dtype=tf.float32)
y_val_tf = tf.convert_to_tensor(y_val, dtype=tf.float32)

In [13]:
def EarlyStopping(model, best_loss, counter, best_weights, min_delta=1e-3):
    val_pred = model(x_val_tf)
    val_loss = tf.reduce_mean(tf.square(val_pred - y_val_tf))
    
    if val_loss < (best_loss - min_delta):
        best_loss = val_loss
        counter = 0
        best_weights = model.get_weights()

    else:
        counter += 1

    return best_loss, counter, best_weights, val_loss

def TrainPINN(model, optimizer, Ld, Lp, patience=300, best_loss=np.inf):
    counter = 0
    best_weights = model.get_weights()

    for epoch in range(20000):

        loss = train_step(model, optimizer, x_train_tf, y_train_tf,
                          Wd_train, We_train, theta_train, Ld, Lp)
        
        best_loss, counter, best_weights, val_loss =  EarlyStopping(model, best_loss, counter, best_weights)
        
        if counter >= patience:
            print(f"Early stopping at epoch {epoch}")
            model.set_weights(best_weights)
            break

        if epoch % 100 == 0:
            print(f"Epoch {epoch} | Train Loss {loss.numpy():.6f} | Val Loss {val_loss.numpy():.6f}")

In [18]:
TITLES = ["Train_1", "Train_2", "Test_1", "Test_2", "Test_3", "Val", "LSG 1", "LSG 2"]

def EvalModel(model):
    n_targets = len(TARGET)

    # estrutura das métricas
    metrics = {
        name: {
            "R2_Train_1": [], "R2_Train_2": [],
            "R2_Val": [],
            "R2_Test_1": [], "R2_Test_2": [], "R2_Test_3": [],
            "MSE_Train_1": [], "MSE_Train_2": [],
            "MSE_Val": [],
            "MSE_Test_1": [], "MSE_Test_2": [], "MSE_Test_3": [],
        }
        for name in TARGET_INT
    }

    for i, dataset in enumerate(NormDatasets):

        x = dataset[PREDICTORS]
        y = dataset[TARGET]

        x, y = CreateSequences(x, y, TIME_STEPS)

        # predição da rede
        pred = model(tf.convert_to_tensor(x, dtype=tf.float32)).numpy()

        # desnormalização
        y_pred_diff = OUT_SCALER.inverse_transform(pred)
        y_diff = OUT_SCALER.inverse_transform(y)

        # arrays reconstruídos
        y_true = np.zeros_like(y_diff)
        y_pred = np.zeros_like(y_pred_diff)
        y_cin = GetCin(Datasets[i])
        
        n = y_true.shape[0]
        y_cin = y_cin[:n]

        # valores iniciais reais
        init_vals = np.array([
            Datasets[i][name].iloc[0] for name in TARGET_INT
        ])

        # reconstrução por integração cumulativa
        for j in range(n_targets):
            y_true[:, j] = init_vals[j] + np.cumsum(y_diff[:, j] * TS)
            y_pred[:, j] = init_vals[j] + np.cumsum(y_pred_diff[:, j] * TS)

        # cálculo das métricas
        for j, name in enumerate(TARGET_INT):

            r2 = r2_score(y_true[:, j], y_pred[:, j])
            mse = mean_squared_error(y_true[:, j], y_pred[:, j])

            metrics[name][f"R2_{TITLES[i]}"].append(r2)
            metrics[name][f"MSE_{TITLES[i]}"].append(mse)

            print(
                f"{name} | {TITLES[i]} -> "
                f"R² = {r2:.4f}, MSE = {mse:.4e}"
            )

    return metrics

In [19]:
def UpdateRow(metrics, arch, Ld, Lp, r, seed, excel_file):

    model_name = f"model_arch{'-'.join(map(str, arch))}_r{r}_seed{seed}"

    row = {
        "model": model_name,
        "Neurons": arch,
        "Ld": Ld,
        "Lp": Lp,
        "reg": r,
    }

    for name in TARGET_INT:
        row.update({
            f"R2_Train_1_{name}": metrics[name]["R2_Train_1"],
            f"MSE_Train_1_{name}": metrics[name]["MSE_Train_1"],
            f"R2_Train_2_{name}": metrics[name]["R2_Train_2"],
            f"MSE_Train_2_{name}": metrics[name]["MSE_Train_2"],
            f"R2_Val_{name}": metrics[name]["R2_Val"],
            f"MSE_Val_{name}": metrics[name]["MSE_Val"],
            f"R2_Test_1_{name}": metrics[name]["R2_Test_1"],
            f"MSE_Test_1_{name}": metrics[name]["MSE_Test_1"],
            f"R2_Test_2_{name}": metrics[name]["R2_Test_2"],
            f"MSE_Test_2_{name}": metrics[name]["MSE_Test_2"],
            f"R2_Test_3_{name}": metrics[name]["R2_Test_3"],
            f"MSE_Test_3_{name}": metrics[name]["MSE_Test_3"],
        })

    df = pd.DataFrame([row])

    try:
        old = pd.read_excel(excel_file)
        new_df = pd.concat([old, df], ignore_index=True)
        new_df.to_excel(excel_file, index=False)
    except FileNotFoundError:
        df.to_excel(excel_file, index=False)

    print(f"Modelo {arch} | Ld={Ld} Lp={Lp} r={r} seed={seed} salvo.")


In [20]:
import os

def ExportModel(pinn_model, model_name):

    os.makedirs("weights", exist_ok=True)
    os.makedirs("models", exist_ok=True)

    rnn_model = PINN_to_RNN(pinn_model, TIME_STEPS, INPUT_SIZE)

    weights_path = f"weights/{model_name}.weights.h5"
    model_path = f"models/{model_name}.keras"

    rnn_model.save(model_path)

    rnn_model.save_weights(weights_path)

    print(f"Modelo salvo em:\n{model_path}\n{weights_path}")

In [21]:
from tensorflow.keras.optimizers import Adam
from itertools import product

N_MODELS = 5
seeds = np.random.choice(range(1, 10000), size=N_MODELS, replace=False)

architectures = [[16, 8, 4]]

Ld_Lp = [[0.3,0.7], [0.7,0.3]]
r_values = [0.01]

results = {}
# produto cartesiano de todos hiperparâmetros
for arch, (Ld, Lp), r in product(architectures, Ld_Lp, r_values):

    for i, s in enumerate(seeds):

        tf.keras.backend.clear_session()

        init = initializers.RandomNormal(seed=int(s))
        reg = tf.keras.regularizers.l2(r)

        pinn_model = PINN(
            architecture=arch,
            initializer=init,
            regularizer=reg
        )
        pinn_model.build((None, TIME_STEPS, INPUT_SIZE))

        opt = Adam(learning_rate=0.001)
        opt.build(pinn_model.trainable_variables)

        TrainPINN(
            pinn_model,
            Ld=Ld,
            Lp=Lp,
            optimizer=opt
        )
        ExportModel(pinn_model, model_name=f"model_arch{'-'.join(map(str, arch))}_r{r}_seed{s}")
        metrics = EvalModel(pinn_model)
        UpdateRow(metrics, arch, Ld, Lp, r, s, excel_file="resultados.xlsx")
        

Epoch 0 | Train Loss 1.062328 | Val Loss 0.840849
Epoch 100 | Train Loss 0.867438 | Val Loss 0.573609
Epoch 200 | Train Loss 0.818003 | Val Loss 0.556138
Epoch 300 | Train Loss 0.745242 | Val Loss 0.534946
Epoch 400 | Train Loss 0.717567 | Val Loss 0.522436
Epoch 500 | Train Loss 0.699784 | Val Loss 0.498426
Epoch 600 | Train Loss 0.666097 | Val Loss 0.530951
Epoch 700 | Train Loss 0.643422 | Val Loss 0.613415
Epoch 800 | Train Loss 0.623790 | Val Loss 0.721586
Early stopping at epoch 813
Modelo salvo em:
models/model_arch16-8-4_r0.01_seed1133.keras
weights/model_arch16-8-4_r0.01_seed1133.weights.h5
Theta | Train_1 -> R² = 0.8658, MSE = 3.0627e-02
Theta | Train_2 -> R² = 0.7100, MSE = 6.6213e-02
Theta | Test_1 -> R² = 0.8312, MSE = 3.8776e-02
Theta | Test_2 -> R² = 0.8450, MSE = 3.6753e-02
Theta | Test_3 -> R² = 0.3697, MSE = 2.0722e-01
Theta | Val -> R² = -1.4100, MSE = 7.8874e-01
Modelo [16, 8, 4] | Ld=0.3 Lp=0.7 r=0.01 seed=1133 salvo.
Epoch 0 | Train Loss 1.062415 | Val Loss 0.8406